## Импорт библиотек


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Загрузка данных


In [ ]:
data = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(data)


## Признаки и целевая переменная


In [ ]:
features = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year",
]
target = "fuel_efficiency_mpg"

df = df[features + [target]].copy()
print(df.shape)
print(df.head())


## Распределение топливной эффективности


In [ ]:
df[target].hist(bins=50, figsize=(8, 4))
plt.xlabel("Топливная эффективность, MPG")
plt.ylabel("Число автомобилей")
plt.title("Распределение fuel_efficiency_mpg")
plt.tight_layout()
plt.show()

print(df[target].describe())
print("Асимметрия:", round(df[target].skew(), 3))


## Вопрос 1 — пропуски


In [ ]:
missing = df.isna().sum()
print(missing)
missing_column = missing[missing > 0].index[0]
print("Ответ:", missing_column)


## Вопрос 2 — медиана horsepower


In [ ]:
horsepower_median = df["horsepower"].median()
print("Ответ:", horsepower_median)


## Разделение на train, validation и test


In [ ]:
def split_data(df, seed=42):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]].copy().reset_index(drop=True)
    df_val = df.iloc[idx[n_train:n_train + n_val]].copy().reset_index(drop=True)
    df_test = df.iloc[idx[n_train + n_val:]].copy().reset_index(drop=True)

    return df_train, df_val, df_test

df_train, df_val, df_test = split_data(df, seed=42)
print("Train:", len(df_train), "Validation:", len(df_val), "Test:", len(df_test))


## Целевая переменная без логарифмирования


In [ ]:
y_train = df_train[target].to_numpy()
y_val = df_val[target].to_numpy()
y_test = df_test[target].to_numpy()


## Линейная регрессия


In [ ]:
def train_linear_regression(X, y):
    X = np.column_stack([np.ones(X.shape[0]), X])
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]


## Метрика RMSE


In [ ]:
def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())


## Подготовка признаков: prepare_X


In [ ]:
def prepare_X(df, fill_value=0):
    return df[features].fillna(fill_value).to_numpy(dtype=float)


## Вопрос 3 — ноль или среднее


In [ ]:
horsepower_mean = df_train["horsepower"].mean()
imputation_rows = []

for method, fill_value in [("Ноль", 0), ("Среднее", horsepower_mean)]:
    X_train = prepare_X(df_train, fill_value)
    X_val = prepare_X(df_val, fill_value)

    w0, w = train_linear_regression(X_train, y_train)
    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)

    imputation_rows.append({
        "Заполнение": method,
        "RMSE": score,
        "RMSE (3 знака)": round(score, 3),
    })

imputation_results = pd.DataFrame(imputation_rows)
print(imputation_results.to_string(index=False))
rounded_scores = imputation_results["RMSE (3 знака)"]
q3_answer = (
    "Оба варианта одинаково хороши"
    if rounded_scores.nunique() == 1
    else imputation_results.loc[rounded_scores.idxmin(), "Заполнение"]
)
print("Ответ:", q3_answer)


## Линейная регрессия с регуляризацией


In [ ]:
def train_linear_regression_reg(X, y, r=0.001):
    X = np.column_stack([np.ones(X.shape[0]), X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]


## Вопрос 4 — подбор регуляризации


In [ ]:
X_train = prepare_X(df_train, fill_value=0)
X_val = prepare_X(df_val, fill_value=0)
regularization_rows = []

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    score = rmse(y_val, w0 + X_val.dot(w))
    regularization_rows.append({
        "r": r,
        "RMSE": score,
        "RMSE (4 знака)": round(score, 4),
    })

regularization_results = pd.DataFrame(regularization_rows)
best_r = regularization_results.sort_values(
    ["RMSE (4 знака)", "r"]
).iloc[0]["r"]

print(regularization_results.to_string(index=False))
print("Ответ:", best_r)


## Вопрос 5 — влияние seed


In [ ]:
seed_rows = []

for seed in range(10):
    train_seed, val_seed, _ = split_data(df, seed=seed)
    X_train_seed = prepare_X(train_seed, fill_value=0)
    X_val_seed = prepare_X(val_seed, fill_value=0)
    y_train_seed = train_seed[target].to_numpy()
    y_val_seed = val_seed[target].to_numpy()

    w0_seed, w_seed = train_linear_regression(X_train_seed, y_train_seed)
    score = rmse(y_val_seed, w0_seed + X_val_seed.dot(w_seed))
    seed_rows.append({"seed": seed, "RMSE": score})

seed_results = pd.DataFrame(seed_rows)
std = np.std(seed_results["RMSE"].to_numpy())

print(seed_results.to_string(index=False))
print("Стандартное отклонение:", std)
print("Ответ:", round(std, 3))


## Вопрос 6 — объединение train и validation, seed 9


In [ ]:
df_train_9, df_val_9, df_test_9 = split_data(df, seed=9)

df_full_train = pd.concat([df_train_9, df_val_9], ignore_index=True)
y_full_train = df_full_train[target].to_numpy()
y_test_9 = df_test_9[target].to_numpy()

X_full_train = prepare_X(df_full_train, fill_value=0)
X_test_9 = prepare_X(df_test_9, fill_value=0)


## Вопрос 6 — финальная модель и тест


In [ ]:
w0_final, w_final = train_linear_regression_reg(
    X_full_train, y_full_train, r=0.001
)
y_pred_test = w0_final + X_test_9.dot(w_final)
test_rmse = rmse(y_test_9, y_pred_test)

print("Test RMSE:", test_rmse)
print("Ответ:", round(test_rmse, 3))


## Предсказание для одного автомобиля


In [ ]:
car = df_test_9[features].iloc[20].to_dict()
X_car = prepare_X(pd.DataFrame([car]), fill_value=0)

predicted_mpg = float(w0_final + X_car.dot(w_final)[0])
actual_mpg = float(y_test_9[20])

print("Автомобиль:", car)
print("Прогноз, MPG:", round(predicted_mpg, 3))
print("Фактическое значение, MPG:", actual_mpg)


## Ответы


In [ ]:
answers = pd.DataFrame({
    "Вопрос": [1, 2, 3, 4, 5, 6],
    "Ответ": [
        missing_column,
        horsepower_median,
        q3_answer,
        best_r,
        round(std, 3),
        round(test_rmse, 3),
    ],
})
print(answers.to_string(index=False))
